# Content-Based Book Recommender with ChromaDB

We build a content-based recommender over ~1,200 book summaries. Instead of materialising a
full book-to-book similarity matrix with `pairwise_distances`, we store the summary embeddings
in a **ChromaDB** vector store and let it do the nearest-neighbour search for us.

Two retrieval paths are supported:

1. **`get_similar_books(title)`** - "more books like *this* book". The book's own embedding is
   pulled from the store and used as the query vector.
2. **`search_books(query)`** - "find me books about *this* idea". A free-text keyword or phrase
   typed by the user is embedded on the fly and used as the query vector.

Both paths hit the same collection. That is the whole point of a vector store: a query is just
another vector, whether it came from a document or from a user.

*Manaranjan Pradhan / AIThinking.in*

## Setup

In [1]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Loading the Book Dataset

In [3]:
import pandas as pd
import numpy as np
import ast

In [4]:
# Set the maximum column width to 100 characters
pd.set_option('display.max_colwidth', 500)

In [5]:
books_df = pd.read_csv('books.csv', index_col=[0])

In [6]:
books_df.head(10)

,book_name,summaries,categories
0,The Highly Sensitive Person,"is a self-assessment guide and how-to-live template for people who feel, relate, process, and notice more deeply than others, and who frequently suffer from overstimulation as a result.",science
1,Why Has Nobody Told Me This Before?,"is a collection of a clinical psychologist’s best practical advice to combat anxiety and depression and improve our mental health in small increments, collected from over a decade of 1-on-1 work with patients.",science
2,The Midnight Library,"tells the story of Nora, a depressed woman in her 30s, who, on the day she decides to die, finds herself in a library full of lives she could have lived, where she discovers there’s a lot more to life, even her current one, than she had ever imagined.",science
3,Brave New World,"presents a futuristic society engineered perfectly around capitalism and scientific efficiency, in which everyone is happy, conform, and content — but only at first glance.",science
4,1984,is the story of a man questioning the system that keeps his futuristic but dystopian society afloat and the chaos that quickly ensues once he gives in to his natural curiosity and desire to be free.,science
5,Stolen Focus,"explains why our attention spans have been dwindling for decades, how technology accelerates this worrying trend, and what we can do to reclaim our focus and thus our capacity to live meaningful lives.",science
6,The Life-Changing Science of Detecting Bullshit,teaches its readers how to avoid falling for the lies and false information that other people spread by helping them build essential thinking skills through examples from the real world.,science
7,Dopamine Nation,"talks about the importance of living a balanced life in relation to all the pleasure and stimuli we’re surrounded with on a daily basis, such as drugs, devices, porn, gambling facilities, showing us how to avoid becoming dopamine addicts by restricting our access to them.",science
8,The Art of Statistics,"is a non-technical book that shows how statistics is helping humans everywhere get a new hold of data, interpret numbers, fact-check information, and reveal valuable insights, all while keeping the world as we know it afloat.",science
9,No Self No Problem,"is a provocative read about the implications of Buddhism in neuroscience, and more specifically about the idea that the self is only a product of the mind, meaning that there is no “I”.",science


In [7]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1227 entries, 0 to 1226
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   book_name   1227 non-null   object
 1   summaries   1227 non-null   object
 2   categories  1227 non-null   object
dtypes: object(3)
memory usage: 38.3+ KB


In [8]:
books_df.shape

(1227, 3)

In [9]:
books_df.categories.value_counts()

,count
categories,
happiness,218
relationships,204
science,199
productivity,168
politics,80
biography,69
money,62
psychology,43
economics,39


## Setting up the ChromaDB Vector Store

Three moving parts:

- **Embedding function** - `all-MiniLM-L6-v2`, a 384-dimensional sentence transformer. Chroma calls
  it for us on both the documents we add and the queries we run, so the two always land in the same
  space.
- **Client** - `PersistentClient` writes to disk, so the embeddings survive a kernel restart. Use
  `chromadb.EphemeralClient()` if you want a purely in-memory store.
- **Collection** - the table. We set the distance metric to cosine, which is what we were computing
  by hand earlier.

In [10]:
import chromadb
from chromadb.utils import embedding_functions

In [11]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
chroma_client = chromadb.PersistentClient(path = "./chroma_books_db")

In [13]:
book_collection = chroma_client.get_or_create_collection(
    name = "books",
    embedding_function = embedding_fn,
    metadata = {"hnsw:space": "cosine"}
)

### Loading the summaries into the collection

Every record needs three things:

- `ids` - our key back into `books_df`. We use the dataframe index as a string.
- `documents` - the raw summary text. Chroma embeds it for us.
- `metadatas` - anything we want to filter on or display later, here the book name and category.

We add in batches and skip the work if the collection is already populated, so re-running the
notebook does not re-embed 1,227 summaries.

In [14]:
def load_books_into_chroma(df, collection, batch_size = 200):
    if collection.count() == len(df):
        print(f"Collection already has {collection.count()} books. Skipping load.")
        return

    ids       = [str(idx) for idx in df.index]
    documents = df['summaries'].tolist()
    metadatas = [{"book_name": row.book_name, "categories": row.categories}
                 for row in df.itertuples()]

    for start in range(0, len(df), batch_size):
        end = start + batch_size
        collection.add(ids       = ids[start:end],
                       documents = documents[start:end],
                       metadatas = metadatas[start:end])
        print(f"Added books {start} to {min(end, len(df)) - 1}")

    print(f"Done. Collection size: {collection.count()}")

In [15]:
load_books_into_chroma(books_df, book_collection)

Added books 0 to 199
Added books 200 to 399
Added books 400 to 599
Added books 600 to 799
Added books 800 to 999
Added books 1000 to 1199
Added books 1200 to 1226
Done. Collection size: 1227


In [16]:
book_collection.count()

1227

### What does a stored record look like?

`peek()` returns the first few records. Note the embedding: 384 dense floats, in contrast to the
900 mostly-zero TF-IDF columns.

In [17]:
sample_record = book_collection.peek(limit = 2)

print("Keys stored     :", [k for k, v in sample_record.items() if v is not None])
print("IDs             :", sample_record['ids'])
print("Metadata        :", sample_record['metadatas'])
print("Embedding shape :", sample_record['embeddings'].shape)
print("First 8 dims    :", sample_record['embeddings'][0][:8].round(3))

Keys stored     : ['ids', 'embeddings', 'documents', 'included', 'metadatas']
IDs             : ['0', '1']
Metadata        : [{'book_name': 'The Highly Sensitive Person', 'categories': 'science'}, {'book_name': 'Why Has Nobody Told Me This Before?', 'categories': 'science'}]
Embedding shape : (2, 384)
First 8 dims    : [-0.005  0.065  0.015  0.078  0.007 -0.008 -0.011  0.07 ]


## A Helper to Format Chroma's Results

Chroma returns a dict of parallel lists, one entry per query. Since we always send a single query,
we take element `[0]` of each list and flatten it into a dataframe.

Chroma reports **cosine distance**, so we convert it back to the similarity score we are used to:

$$\text{similarity} = 1 - \text{cosine distance}$$

In [18]:
def to_dataframe(results):
    records = []

    for book_id, meta, doc, dist in zip(results['ids'][0],
                                        results['metadatas'][0],
                                        results['documents'][0],
                                        results['distances'][0]):
        records.append({
            "book_id"    : int(book_id),
            "book_name"  : meta['book_name'],
            "categories" : meta['categories'],
            "summaries"  : doc,
            "similarity" : round(1 - dist, 4)
        })

    return pd.DataFrame(records)

## Path 1: Finding TopN Similar Books (book as the query)

Same idea as before, but no similarity matrix. We look up the book's stored embedding with `get()`
and hand that vector straight back to `query()`.

We ask for `topN + 1` neighbours because the closest match to a book is always itself, and then
drop it. The optional `category` argument shows off Chroma's metadata filter - the search runs only
over the books that match, instead of ranking everything and filtering afterwards.

In [19]:
def get_similar_books(title, topN = 5, category = None):
    # Resolve the title: exact match first, then a case-insensitive partial match
    matches = books_df[books_df['book_name'] == title]

    if len(matches) == 0:
        matches = books_df[books_df['book_name'].str.contains(title, case = False, regex = False)]

    if len(matches) == 0:
        raise ValueError(f"No book found matching '{title}'")

    book_idx  = matches.index[0]
    book_name = books_df.loc[book_idx, 'book_name']

    # Pull the embedding Chroma already computed for this book
    stored = book_collection.get(ids = [str(book_idx)], include = ["embeddings"])
    book_embedding = stored['embeddings']

    where_filter = {"categories": category} if category else None

    results = book_collection.query(
        query_embeddings = book_embedding,
        n_results        = topN + 1,
        where            = where_filter,
        include          = ["documents", "metadatas", "distances"]
    )

    similar_df = to_dataframe(results)

    # The book itself is always the nearest neighbour. Drop it.
    similar_df = similar_df[similar_df.book_id != book_idx].head(topN)

    print(f"Books similar to: {book_name}")
    return similar_df.reset_index(drop = True)

### Finding Similar Books

 - The Miracle of Mindfulness
 - The Bitcoin Standard
 - Mind Gym
 - Get A Financial Life

In [20]:
books_df[books_df.book_name.str.contains("Mindfulness")]

,book_name,summaries,categories
731,The Miracle of Mindfulness,teaches the ancient Buddhist practice of mindfulness and how living in the present will make you happier.,happiness


In [21]:
get_similar_books('The Miracle of Mindfulness')

Books similar to: The Miracle of Mindfulness


,book_id,book_name,categories,summaries,similarity
0,718,Radical Acceptance,happiness,teaches how you can become more content and happy in your life by applying the principles of meditation and Buddhism.,0.8150
1,612,The Art of Living,happiness,"talks about living a peaceful life through meditation and gratitude, especially by using the Vipassana meditation technique and the philosophy behind Buddhism, which promotes developing a clearer vision of life and seeing things as they truly are.",0.7012
2,410,That Sounds Fun,relationships,"uncovers the secrets of a happy life: mindfulness, love, joy, and a good dose of doing whatever makes us happy as often as we can, starting from simple, day-to-day activities, to much bigger life experiences that speak to our soul.",0.6445
3,660,Unplug,happiness,"is your guide to utilizing meditation to enhance your brain, deal with stress, and become happier, explaining the basics of this practice, how to get started with it, and what science has to teach about its many benefits.",0.6286
4,737,Inner Engineering,happiness,is a guide to creating a life of happiness by exploring your internal landscape of thoughts and feelings and learning to align them with what the universe tells you.,0.6273


In [22]:
get_similar_books('The Bitcoin Standard')

Books similar to: The Bitcoin Standard


,book_id,book_name,categories,summaries,similarity
0,370,The Age Of Cryptocurrency,economics,"explains the past, present, and future of Bitcoin, including its benefits and drawbacks, how it aligns with the definition of money well enough to be its own currency, how it and other cryptocurrencies will change our economy and the entire world.",0.7167
1,371,Digital Gold,economics,"details the beginnings of Bitcoin, including how it was developed, why it’s early years were such a struggle, the many people that contributed to its rise, and how it’s changed our world so far and why it will continue doing so for a long time.",0.6613
2,829,Blockchain Revolution,money,"explains how the power of this new technology behind Bitcoin can transform our world financially by improving the way we store our money and do business to make it more fair, transparent, equal, and free from corruption.",0.6381
3,308,Capitalism,politics,shows you how the movement of money in the world really works by outlining the dominant system in the world and its origins and future.,0.6273
4,378,Adaptive Markets,economics,"gives you a better understanding of how the movement of money in the world works by outlining the characteristics of the market, some of which are more like living creatures than you might think.",0.6216


In [23]:
get_similar_books('Mind Gym')

Books similar to: Mind Gym


,book_id,book_name,categories,summaries,similarity
0,891,Chasing Excellence,productivity,"breaks down how world-class athletes achieve the mental strength they need to succeed, highlighting",0.7937
1,958,The Inner Game Of Tennis,productivity,"is about the mental state required to deliver peak performance and how you can cultivate that state in sports, work, and life.",0.6835
2,994,The Rise Of Superman,productivity,"decodes the science of ultimate, human performance by examining how top athletes enter and stay in a state of flow, while achieving their greatest feats, and how you can do the same.",0.6283
3,1064,Psyched Up,psychology,is an in-depth look at the science behind mental preparation that will show you how to do your best when it counts the most based on what top performers do.,0.5942
4,1015,The Art Of Learning,productivity,"explains the science of becoming a top performer, based on Josh Waitzkin’s personal rise to the top of the chess and Tai Chi world, by showing you the right mindset, proper ways to practice and how to build the habits of a professional.",0.5811


In [24]:
get_similar_books('Get A Financial Life')

Books similar to: Get A Financial Life


,book_id,book_name,categories,summaries,similarity
0,866,The Millionaire Next Door,money,shows you the simple spending and saving habits that lead to more cash in the bank than most people earn in their life while helping you avoid critical mistakes on your way to financial independence.,0.6615
1,832,Broke Millennial,money,shows those in their twenties and thirties how to manage their finances so that they can stop scraping by and instead begin to live more confidently when it comes to money.,0.6519
2,852,Dollars And Sense,money,explains why it’s so hard to manage money and teaches you how to combat false cues and natural desires so you can manage your dollars in better ways.,0.6418
3,208,More Money Than God,biography,"teaches us about the ins and outs of hedge funds, how those managing money makes a profit, and how you can learn from them and apply their techniques to your money management strategy.",0.5785
4,697,The Latte Factor,happiness,teaches us how to overcome limiting beliefs about money and build our financial freedom through small daily choices.,0.5647


### Recommending within a category

Passing `category` pushes the constraint down into the vector store, so the nearest-neighbour search
is only performed over books in that category.

In [25]:
get_similar_books('The Bitcoin Standard', topN = 5, category = 'economics')

Books similar to: The Bitcoin Standard


,book_id,book_name,categories,summaries,similarity
0,370,The Age Of Cryptocurrency,economics,"explains the past, present, and future of Bitcoin, including its benefits and drawbacks, how it aligns with the definition of money well enough to be its own currency, how it and other cryptocurrencies will change our economy and the entire world.",0.7167
1,371,Digital Gold,economics,"details the beginnings of Bitcoin, including how it was developed, why it’s early years were such a struggle, the many people that contributed to its rise, and how it’s changed our world so far and why it will continue doing so for a long time.",0.6613
2,378,Adaptive Markets,economics,"gives you a better understanding of how the movement of money in the world works by outlining the characteristics of the market, some of which are more like living creatures than you might think.",0.6216
3,367,Cryptoassets,economics,"is your guide to understanding this revolutionary new digital asset class and explains the history of Bitcoin, how to invest in it and other cryptocurrencies, and how the blockchain technology behind it all works.",0.5582
4,369,Narrative Economics,economics,"explains the influence that popular stories have on the way economies operate, including the rise of Bitcoin, stock market booms and crashes, the nature of epidemics, and more.",0.5440


## Path 2: Searching Books by Keyword or Phrase (text as the query)

This is what the similarity matrix could never do. The user types anything - a keyword, a phrase, a
whole sentence describing what they are in the mood for - and Chroma embeds it with the *same*
model, then finds the summaries closest to it.

Note that nothing about the collection changes. We only swap `query_embeddings` for `query_texts`.
There is no keyword matching happening here: a query can retrieve a book whose summary does not
contain a single word of the query, as long as the meaning lines up.

In [26]:
def search_books(query, topN = 5, category = None):
    where_filter = {"categories": category} if category else None

    results = book_collection.query(
        query_texts = [query],
        n_results   = topN,
        where       = where_filter,
        include     = ["documents", "metadatas", "distances"]
    )

    print(f"Search results for: '{query}'" + (f" (category: {category})" if category else ""))
    return to_dataframe(results)

### Single keyword

In [27]:
search_books("procrastination")

Search results for: 'procrastination'


,book_id,book_name,categories,summaries,similarity
0,820,Eat That Frog,happiness,provides 21 techniques and strategies to stop procrastinating and get more done.,0.7223
1,997,The Now Habit,productivity,"is a strategic program to help you eliminate procrastination from your life, bring fun and motivation back to your work and enjoy your well-earned spare time without feeling guilty.",0.6463
2,967,The 5 Second Rule,productivity,"is a simple tool that undercuts most of the psychological weapons your brain employs to keep you from taking action, which will allow you to procrastinate less, live happier and reach your goals.",0.6069
3,1093,Do The Work,psychology,"is Steven Pressfield’s follow-up to The War Of Art, where he gives you actionable tactics and strategies to overcome resistance, the force behind procrastination.",0.5855
4,44,How To Change,science,"identifies the stumbling blocks that are in your way of reaching your goals and improving yourself and the research-backed ways to get over them, including how to beat some of the worst productivity and life problems like procrastination, laziness, and much more.",0.5497


### A phrase

In [28]:
search_books("how to build good habits and break bad ones")

Search results for: 'how to build good habits and break bad ones'


,book_id,book_name,categories,summaries,similarity
0,1222,Better Than Before,work,"breaks down the latest research on how to break bad habits and develop good ones, in order to help you find your habit tendency and give you a few simple tools to start improving your own habits.",0.7837
1,1051,You Are Not Your Brain,productivity,"educates you about the science behind bad habits and breaking them, giving you an actionable 4-step framework you can use to stop listening to your brain’s deceptive messages.",0.6875
2,139,Atomic Habits,science,"is the definitive guide to breaking bad behaviors and adopting good ones in four steps, showing you how small, incremental, everyday routines compound into massive, positive change over time.",0.6626
3,1016,Carrots And Sticks,productivity,"explains how you can harness the power of incentives – carrots and sticks – to change your bad behaviors, improve your self-control and reach your long-term goals.",0.5911
4,804,Rewire,happiness,"explains why we keep engaging in addictive and self-destructive behavior, how our brains justify it and where you can get started on breaking your bad habits by becoming more mindful and disciplined.",0.5826


### A full sentence describing an intent

Watch the top results here. The user never says *"negotiation"* or *"persuasion"*, but the retrieved
books are about exactly that. This is the payoff of semantic embeddings over TF-IDF's exact-token
matching.

In [29]:
search_books("I want to become better at convincing people in difficult conversations")

Search results for: 'I want to become better at convincing people in difficult conversations'


,book_id,book_name,categories,summaries,similarity
0,418,I Hear You,relationships,"explores the idea of becoming a better listener, engaging in productive conversations and avoiding building up frustrations by taking charge of your communication patterns and improving them in your further dialogues.",0.6536
1,413,"How to Talk to Anyone, Anytime, Anywhere",relationships,"shares the secrets of effective communication and teaches you how to adopt a charisma that’ll help you navigate conversations easier, break the ice, and get your point across right away, and talk to people in general.",0.6360
2,1100,Words That Work,motivation,"outlines the importance of using the right words and the appropriate body language in a given situation to make yourself understood properly and get the most out of the dialogue, while also teaching you some tips-and-tricks on how to win arguments, tame conflicts, and get your point across using a wise selection of words.",0.6341
3,526,Just Listen,relationships,teaches how to get your message across to anyone by using proven listening and persuasion techniques.,0.6028
4,513,The Fine Art Of Small Talk,relationships,"will teach you how to skillfully start, continue, and end conversations with anyone, no matter how shy you think you are.",0.5723


### Search restricted to a category

In [30]:
search_books("saving and investing for retirement", topN = 5, category = "money")

Search results for: 'saving and investing for retirement' (category: money)


,book_id,book_name,categories,summaries,similarity
0,865,Rule #1,money,"hands you the reins of personal investing, even if you’ve never held them before, by using a few simple rules from Warren Buffett’s value investing approach to guide you towards financial independence.",0.4376
1,871,The Little Book of Common Sense Investing,money,"shows you an alternative to actively, poorly managed, overpaid funds by introducing you to low-cost, passive index funds as a sustainable investing strategy, which gets you the retirement savings you need without the usual hassle of stock investing.",0.4213
2,834,From Here To Financial Happiness,money,is your guide to having a healthier relationship with money and shows you how looking to the future to prepare for what’s ahead can make you a lot less stressed about your financial life.,0.4197
3,873,Secrets Of The Millionaire Mind,money,suggests our financial success is predetermined from birth and shows us what to do to break through mental barriers and acquire the habits and thinking of the rich.,0.3984
4,866,The Millionaire Next Door,money,shows you the simple spending and saving habits that lead to more cash in the bank than most people earn in their life while helping you avoid critical mistakes on your way to financial independence.,0.3945


### Housekeeping

Chroma persisted everything to `./chroma_books_db`. Uncomment below to wipe the collections and
start over.

In [ ]:
# chroma_client.delete_collection("books")
# chroma_client.delete_collection("books_tfidf")
# chroma_client.list_collections()